# Sesión 7 · Resultados predictivos y KPIs con Dash
**Misión 9–11 nov 2026 · Miércoles 11 · 11:00–12:30 · Santiago Quintana**

Notebook de **demostración** (primera mitad, ~45 min). En la segunda mitad: `S7_taller_estudiantes.ipynb`.

**Caso:** con las medidas de un pingüino, ¿podemos predecir su **sexo**? (1 = Male, 0 = Female). No entrenamos modelos aquí: partimos de un **CSV de predicciones** y construimos el tablero que comunica los resultados, con la misma estructura de Dash de la sesión 6.

| Bloque | Min | Contenido |
|---|---|---|
| 0. Del tablero descriptivo al predictivo | 5 | Qué cambia y qué se mantiene |
| 1. El CSV de predicciones | 5 | Columnas, carga y validación |
| 2. KPIs: `metricas(df, umbral)` | 10 | Accuracy, precision, recall, F1 |
| 3. Figuras: `fig_confusion` y compañía | 10 | Matriz de confusión, probabilidades, ROC, comparación |
| 4. La app: maqueta + callback | 10 | Modelo, especie y umbral; descarga; resumen ejecutivo |
| 5. Narrativa ejecutiva | 5 | Conclusiones, limitaciones, recomendación |

In [1]:
# En Google Colab: instalar lo que no viene por defecto (en local puede omitirse)
import importlib.util, subprocess, sys
for pkg in ["dash", "plotly"]:
    if importlib.util.find_spec(pkg.replace('-', '_')) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)

In [2]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, roc_curve)
from dash import Dash, html, dcc, Input, Output, State

## 0. Del tablero descriptivo (sesión 6) al predictivo (sesión 7)

| | Descriptivo (sesión 6) | Predictivo (sesión 7) |
|---|---|---|
| Pregunta | ¿Cómo son los datos? | ¿Qué tan bien predice el modelo? |
| Datos | El dataset original | Un CSV de predicciones |
| KPIs | Conteos y promedios | Accuracy, precision, recall, F1 |
| Gráficos | Barras, histogramas | Matriz de confusión, curva ROC |

Lo de la sesión 6 se mantiene: `html.Div` para la maqueta, `dcc.Graph` para los gráficos y callbacks para los filtros.

**Usuario → pregunta → KPI**

| Usuario | Pregunta | KPI | Gráfico |
|---|---|---|---|
| Investigador/a | ¿Con qué frecuencia acierta? | Accuracy | Matriz de confusión |
| Investigador/a | Si dice macho, ¿acierta? | Precision | Matriz de confusión |
| Investigador/a | ¿Detecta a todos los machos? | Recall | Probabilidades + umbral |
| Investigador/a | ¿Qué modelo usar? | F1 por modelo | Barras comparativas |

## 1. El CSV de predicciones

| columna | significado |
|---|---|
| `id` | identificador de la observación |
| `modelo` | nombre del modelo |
| `y_true` | etiqueta real (0 = Female, 1 = Male) |
| `proba` | probabilidad estimada de que sea macho |
| `species` | especie: sirve para filtrar el tablero |

Si tienes un CSV de otro notebook, basta con poner su ruta en `RUTA_PREDICCIONES`. Si no existe, la celda siguiente genera uno **de respaldo** (3 modelos evaluados con validación cruzada).

In [3]:
RUTA_PREDICCIONES = "predicciones_penguins.csv"

def generar_predicciones_respaldo(ruta):
    p = sns.load_dataset("penguins").dropna().reset_index(drop=True)
    p["id"] = p.index
    X = p[["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g", "species", "island"]]
    y = (p["sex"] == "Male").astype(int)
    prep = make_column_transformer(
        (StandardScaler(), ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]),
        (OneHotEncoder(), ["species", "island"]))
    modelos = {"Regresión logística": make_pipeline(prep, LogisticRegression(max_iter=1000)),
               "Random Forest": make_pipeline(prep, RandomForestClassifier(n_estimators=300, random_state=42)),
               "Gradient Boosting": make_pipeline(prep, HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, random_state=42))}
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    filas = []
    for nombre, m in modelos.items():
        proba = cross_val_predict(m, X, y, cv=cv, method="predict_proba")[:, 1]
        filas.append(pd.DataFrame({"id": p["id"], "modelo": nombre, "y_true": y, "proba": proba.round(4),
                                   "species": p["species"], "island": p["island"]}))
    pd.concat(filas).to_csv(ruta, index=False)
    # importancia de variables (Random Forest ajustado con todos los datos, solo descriptivo)
    rf = modelos["Random Forest"].fit(X, y)
    nombres = rf[0].get_feature_names_out()
    (pd.DataFrame({"variable": [n.split("__")[1] for n in nombres], "importancia": rf[-1].feature_importances_})
       .sort_values("importancia").to_csv("importancias_penguins.csv", index=False))

if not os.path.exists(RUTA_PREDICCIONES):
    generar_predicciones_respaldo(RUTA_PREDICCIONES)

def cargar_predicciones(ruta):
    df = pd.read_csv(ruta)
    faltan = {"id", "modelo", "y_true", "proba"} - set(df.columns)
    assert not faltan, f"Al CSV le faltan columnas: {faltan}"
    assert df["proba"].between(0, 1).all(), "proba debe estar entre 0 y 1"
    return df

pred = cargar_predicciones(RUTA_PREDICCIONES)
importancias = pd.read_csv("importancias_penguins.csv") if os.path.exists("importancias_penguins.csv") else None
print(pred.shape); print(pred["modelo"].value_counts().to_string())
pred.head()

(999, 6)
modelo
Regresión logística    333
Random Forest          333
Gradient Boosting      333


,id,modelo,y_true,proba,species,island
0,0,Regresión logística,1,0.5196,Adelie,Torgersen
1,1,Regresión logística,0,0.2340,Adelie,Torgersen
2,2,Regresión logística,0,0.1097,Adelie,Torgersen
3,3,Regresión logística,0,0.3771,Adelie,Torgersen
4,4,Regresión logística,1,0.9229,Adelie,Torgersen


## 2. KPIs: `metricas(df, umbral)`

Misma idea que `kpis(df)` en la sesión 6: una función que recibe las predicciones (ya filtradas) y devuelve los números de las tarjetas.

| KPI | Pregunta que responde |
|---|---|
| Accuracy | ¿Qué % de las predicciones son correctas? |
| Precision | Cuando el modelo dice "macho", ¿qué % de las veces acierta? |
| Recall | De todos los machos reales, ¿qué % detecta? |
| F1 | Un solo número que equilibra precision y recall |

**El umbral** convierte la probabilidad en decisión: si `proba ≥ umbral`, decimos "macho".

In [4]:
def metricas(df, umbral=0.5):
    y = df["y_true"]
    yhat = df["proba"] >= umbral
    return {
      "accuracy": accuracy_score(y, yhat),
      "precision": precision_score(y, yhat, zero_division=0),
      "recall": recall_score(y, yhat, zero_division=0),
      "f1": f1_score(y, yhat, zero_division=0)}

tabla_metricas = pd.DataFrame({m: metricas(g) for m, g in pred.groupby("modelo")}).T
tabla_metricas.round(3)

,accuracy,precision,recall,f1
Gradient Boosting,0.907,0.915,0.899,0.907
Random Forest,0.907,0.920,0.893,0.906
Regresión logística,0.907,0.905,0.911,0.908


In [5]:
# El umbral mueve precision y recall en direcciones opuestas
lr = pred[pred["modelo"] == "Regresión logística"]
pd.DataFrame({u: metricas(lr, u) for u in [0.3, 0.5, 0.7]}).T.round(2)

,accuracy,precision,recall,f1
0.3,0.90,0.86,0.97,0.91
0.5,0.91,0.91,0.91,0.91
0.7,0.88,0.96,0.80,0.87


## 3. Figuras: cada una es una función que devuelve una figura
Igual que en la sesión 6: luego el callback solo vuelve a llamarlas con los datos filtrados.

**Matriz de confusión** en 3 pasos: `confusion_matrix` cuenta aciertos y errores → `px.imshow` dibuja esa tabla → `dcc.Graph` la pone en la página.

In [6]:
def fig_confusion(df, umbral):
    y = df["y_true"]
    yhat = df["proba"] >= umbral
    cm = confusion_matrix(y, yhat, labels=[False, True])     # 1) contar
    fig = px.imshow(cm, text_auto=True,                        # 2) dibujar
        x=["Pred: Female", "Pred: Male"],
        y=["Real: Female", "Real: Male"],
        color_continuous_scale="Blues", title=f"Matriz de confusión (umbral {umbral:.2f})")
    return fig.update_layout(coloraxis_showscale=False, margin=dict(t=50, l=10, r=10, b=10))

fig_confusion(lr, 0.5)

In [7]:
COLOR_MODELO = {"Regresión logística": "#1F77B4", "Random Forest": "#2CA02C", "Gradient Boosting": "#D62728"}

def fig_probas(df, umbral):
    fig = px.histogram(df.assign(real=df["y_true"].map({0: "Female", 1: "Male"})), x="proba", color="real",
                       nbins=25, barmode="overlay", opacity=0.65, title="¿Qué tan seguro está el modelo?",
                       labels={"proba": "Probabilidad estimada de Male", "real": "Sexo real"})
    fig.add_vline(x=umbral, line_dash="dash", line_color="red")
    return fig.update_layout(margin=dict(t=50, l=10, r=10, b=10))

def fig_roc(df, umbral):
    fpr, tpr, thr = roc_curve(df["y_true"], df["proba"])
    i = np.argmin(np.abs(thr - umbral))
    fig = go.Figure([go.Scatter(x=fpr, y=tpr, mode="lines", name="ROC"),
                     go.Scatter(x=[0, 1], y=[0, 1], mode="lines", line=dict(dash="dash", color="gray"), name="azar"),
                     go.Scatter(x=[fpr[i]], y=[tpr[i]], mode="markers", marker=dict(size=12, color="red"), name="umbral actual")])
    return fig.update_layout(title=f"Curva ROC · AUC = {roc_auc_score(df['y_true'], df['proba']):.3f}",
                             xaxis_title="Tasa de falsos positivos", yaxis_title="Recall", margin=dict(t=50, l=10, r=10, b=10))

def fig_comparacion(df_todos_modelos, umbral):
    m = pd.DataFrame({mod: metricas(g, umbral) for mod, g in df_todos_modelos.groupby("modelo")}).T
    largo = m[["precision", "recall", "f1"]].reset_index(names="modelo").melt(id_vars="modelo", var_name="métrica")
    fig = px.bar(largo, x="métrica", y="value", color="modelo", barmode="group", text_auto=".2f",
                 color_discrete_map=COLOR_MODELO, range_y=[0, 1], title="Comparación de modelos")
    return fig.update_layout(yaxis_title="", margin=dict(t=50, l=10, r=10, b=10))

def fig_importancia():
    if importancias is None:
        return go.Figure().update_layout(title="Importancia no disponible")
    return px.bar(importancias, x="importancia", y="variable", orientation="h",
                  title="Importancia de variables (Random Forest)").update_layout(margin=dict(t=50, l=10, r=10, b=10))

fig_comparacion(pred, 0.5)

**Curva ROC:** muestra qué tan bien separa el modelo las dos clases para todos los umbrales. Cuanto más se acerque a la esquina superior izquierda, mejor; la diagonal gris es adivinar al azar.

In [8]:
fig_roc(lr, 0.5)

## 4. La app: maqueta + callback

**Maqueta** (cada caja es un `html.Div`, como en la sesión 6):
```
html.Div  ← página
├── html.H1  título
└── dcc.Tabs
    ├── Pestaña "Desempeño detallado"
    │   ├── fila de controles: modelo · especie · umbral · descargar
    │   ├── fila de 7 tarjetas KPI
    │   ├── fila: matriz de confusión · comparación de modelos
    │   ├── fila: probabilidades · curva ROC
    │   └── fila: importancia de variables
    └── Pestaña "Resumen ejecutivo": 3 conclusiones, limitaciones, recomendación
```

In [9]:
# Estilos reutilizables (CSS escrito como diccionarios de Python)
COLOR = {"fondo": "#F5F7FA", "texto": "#1F2937", "primario": "#1F4E79", "suave": "#6B7280"}
ESTILO_PAGINA = {"fontFamily": "Arial, sans-serif", "backgroundColor": COLOR["fondo"],
                 "padding": "20px", "color": COLOR["texto"]}
ESTILO_TARJETA = {"backgroundColor": "white", "borderRadius": "10px", "padding": "14px 18px",
                  "boxShadow": "0 1px 4px rgba(0,0,0,.08)", "flex": "1", "minWidth": "160px",
                  "boxSizing": "border-box"}
ESTILO_FILA = {"display": "flex", "gap": "16px", "flexWrap": "wrap", "marginBottom": "16px"}
ESTILO_CAJA = {"backgroundColor": "white", "borderRadius": "10px", "padding": "8px",
               "boxShadow": "0 1px 4px rgba(0,0,0,.08)", "flex": "1", "minWidth": "380px"}

def tarjeta(titulo, valor, id_valor=None):
    """Tarjeta KPI: título pequeño + valor grande. Con id_valor, el valor se puede actualizar con un callback."""
    props = {"id": id_valor} if id_valor else {}
    return html.Div([
        html.Div(titulo, style={"fontSize": "13px", "color": COLOR["suave"]}),
        html.Div(valor, style={"fontSize": "28px", "fontWeight": "700", "color": COLOR["primario"]}, **props),
    ], style=ESTILO_TARJETA)

In [10]:
MODELOS = sorted(pred["modelo"].unique())
ESPECIES = ["Todas"] + sorted(pred["species"].unique())
mejor = tabla_metricas["f1"].idxmax()

def filtrar(pred, modelo, especie):
    df = pred[pred["modelo"] == modelo]
    if especie != "Todas":
        df = df[df["species"] == especie]
    return df

def caja(id_):
    return html.Div(dcc.Graph(id=id_), style=ESTILO_CAJA)

controles = html.Div([
    html.Div([html.Label("Modelo"), dcc.Dropdown(id="f-modelo", options=MODELOS, value=mejor, clearable=False)],
             style={"flex": "1"}),
    html.Div([html.Label("Especie"), dcc.Dropdown(id="f-especie", options=ESPECIES, value="Todas", clearable=False)],
             style={"flex": "1"}),
    html.Div([html.Label("Umbral de decisión"),
              dcc.Slider(id="f-umbral", min=0.1, max=0.9, step=0.05, value=0.5,
                         marks={v: f"{v:.1f}" for v in [0.1, 0.3, 0.5, 0.7, 0.9]})], style={"flex": "2"}),
    html.Button("Descargar CSV", id="btn-csv", style={"height": "36px"}), dcc.Download(id="descarga"),
], style={**ESTILO_TARJETA, "display": "flex", "gap": "20px", "alignItems": "end", "marginBottom": "16px"})

pestana_desempeno = html.Div([
    controles,
    html.Div([tarjeta("n", "", "k-n"), tarjeta("% machos reales", "", "k-tasa"), tarjeta("Accuracy", "", "k-acc"),
              tarjeta("Precision", "", "k-prec"), tarjeta("Recall", "", "k-rec"), tarjeta("F1", "", "k-f1"),
              tarjeta("ROC-AUC", "", "k-auc")], style=ESTILO_FILA),
    html.Div([caja("g-conf"), caja("g-comp")], style=ESTILO_FILA),
    html.Div([caja("g-prob"), caja("g-roc")], style=ESTILO_FILA),
    html.Div([html.Div(dcc.Graph(figure=fig_importancia()), style=ESTILO_CAJA)], style=ESTILO_FILA),
], style={"paddingTop": "16px"})

In [11]:
# Resumen ejecutivo: conclusiones calculadas a partir de los datos, no escritas "a mano"
m_mejor = tabla_metricas.loc[mejor]
f1_especie = {e: metricas(filtrar(pred, mejor, e))["f1"] for e in ESPECIES[1:]}
peor, mejor_esp = min(f1_especie, key=f1_especie.get), max(f1_especie, key=f1_especie.get)
m03, m07 = metricas(filtrar(pred, mejor, "Todas"), 0.3), metricas(filtrar(pred, mejor, "Todas"), 0.7)

conclusiones = [
    (f"Los {len(MODELOS)} modelos prácticamente empatan (F1 entre {tabla_metricas.f1.min():.3f} y {tabla_metricas.f1.max():.3f}); "
     f"conviene preferir el más simple e interpretable."
     if tabla_metricas.f1.max() - tabla_metricas.f1.min() < 0.02 else
     f"{mejor} es el mejor modelo: F1 = {m_mejor.f1:.2f} con umbral 0,5."),
    f"El desempeño cambia por especie: F1 va de {f1_especie[peor]:.2f} ({peor}) a {f1_especie[mejor_esp]:.2f} ({mejor_esp}).",
    f"El umbral es una decisión de negocio: con 0,3 el recall sube a {m03['recall']:.0%} (precision {m03['precision']:.0%}); "
    f"con 0,7 la precision llega a {m07['precision']:.0%} (recall {m07['recall']:.0%}).",
]
pestana_resumen = html.Div([
    html.H2("3 conclusiones"), html.Ol([html.Li(c, style={"marginBottom": "8px"}) for c in conclusiones]),
    html.H3("Limitaciones"),
    html.Ul([html.Li("Muestra pequeña (333 pingüinos): las métricas por especie son menos estables."),
             html.Li("Métricas calculadas con validación cruzada, no con datos de otra temporada."),
             html.Li("La importancia de variables indica asociación, no causalidad.")]),
    html.H3("Recomendación"),
    html.P(f"Usar {mejor} con umbral 0,5; revisar a mano los casos con probabilidad entre 0,4 y 0,6."),
], style={**ESTILO_TARJETA, "maxWidth": "900px", "marginTop": "16px"})
for c in conclusiones: print("•", c)

• Los 3 modelos prácticamente empatan (F1 entre 0.906 y 0.908); conviene preferir el más simple e interpretable.
• El desempeño cambia por especie: F1 va de 0.88 (Adelie) a 0.94 (Gentoo).
• El umbral es una decisión de negocio: con 0,3 el recall sube a 97% (precision 86%); con 0,7 la precision llega a 96% (recall 80%).


In [12]:
app = Dash(__name__)
app.layout = html.Div([
    html.H1("Predicción del sexo de pingüinos · Tablero de resultados"),
    html.P(f"Fuente: {RUTA_PREDICCIONES} · {pred['id'].nunique()} pingüinos · {len(MODELOS)} modelos",
           style={"color": COLOR["suave"]}),
    dcc.Tabs([dcc.Tab(label="Desempeño detallado", children=pestana_desempeno),
              dcc.Tab(label="Resumen ejecutivo", children=pestana_resumen)]),
], style=ESTILO_PAGINA)

def pct(x):
    return "—" if pd.isna(x) else f"{100 * x:.1f} %"

@app.callback(
    Output("k-n", "children"), Output("k-tasa", "children"), Output("k-acc", "children"),
    Output("k-prec", "children"), Output("k-rec", "children"), Output("k-f1", "children"),
    Output("k-auc", "children"),
    Output("g-conf", "figure"), Output("g-comp", "figure"), Output("g-prob", "figure"), Output("g-roc", "figure"),
    Input("f-modelo", "value"), Input("f-especie", "value"), Input("f-umbral", "value"),
)
def actualizar(modelo, especie, umbral):
    df = filtrar(pred, modelo, especie)
    m = metricas(df, umbral)
    todos = pred if especie == "Todas" else pred[pred["species"] == especie]   # para comparar modelos
    return (f"{len(df)}", pct(df["y_true"].mean()), pct(m["accuracy"]), pct(m["precision"]),
            pct(m["recall"]), f"{m['f1']:.2f}", f"{roc_auc_score(df['y_true'], df['proba']):.3f}",
            fig_confusion(df, umbral), fig_comparacion(todos, umbral), fig_probas(df, umbral), fig_roc(df, umbral))

@app.callback(Output("descarga", "data"), Input("btn-csv", "n_clicks"),
              State("f-modelo", "value"), State("f-especie", "value"), State("f-umbral", "value"),
              prevent_initial_call=True)
def descargar(_, modelo, especie, umbral):
    df = filtrar(pred, modelo, especie).assign(y_pred=lambda x: (x["proba"] >= umbral).astype(int))
    return dcc.send_data_frame(df.to_csv, f"predicciones_{modelo}_{especie}.csv", index=False)

# Prueba sin servidor: todas las especies
for e in ESPECIES:
    salida = actualizar(mejor, e, 0.5)
print("Callback OK ·", salida[:7])

Callback OK · ('119', '51.3 %', '93.3 %', '92.1 %', '95.1 %', '0.94', '0.987')


In [ ]:
app.run(jupyter_mode="inline", port=8071, height=1500)

> **Probar en vivo:** mueve el umbral a 0,3 y a 0,7 y mira la matriz de confusión, las tarjetas de precision y recall y la línea roja del histograma. Luego elige una especie: ¿dónde se equivoca más el modelo?

## 5. Narrativa ejecutiva: guía rápida
- **Priorizar KPIs**: el usuario ejecutivo quiere 1 a 3 números y una recomendación.
- **Títulos con hallazgo**: "el modelo detecta 9 de cada 10 machos", no "Recall por modelo".
- **Errores en lenguaje del usuario**: un falso positivo = "dijimos macho y era hembra".
- **Limitaciones explícitas** y **segmentos** donde el modelo falla más.
- **Recomendación accionable**: qué modelo, qué umbral y qué revisar a mano.

**Para correrlo fuera del notebook:** igual que en la sesión 6, todo en un `app.py` con `if __name__ == "__main__": app.run(debug=True)` y luego `python app.py`.

➡️ **Ahora: `S7_taller_estudiantes.ipynb` (45 min).**